# G19_G20_Comparison - Gemma (`google/gemma-2b-it`) vs DeepSeek-R1-Distill-Qwen-7B
Separate comparative assignment: **A. research papers**, **B. technical/implementation**, **C. experiments**.
Part C **consumes the real result files** written by `G19_Gemma_Implementation.ipynb` and `G20_DeepSeek_R1_Implementation.ipynb`
(`G19_results.json`, `G20_results.json`). If a file is missing the notebook says **NOT RUN** and shows no numbers - it never fills gaps.

**Where to put the files:** upload the two `*_results.json` (and optionally the zips) to Colab, or mount Drive and set `SEARCH_DIRS`. Charts are saved to `/content/Pretrained_Model_CA/Comparative_Study/charts/`.

## Setup

In [ ]:
import os, json, glob, math, statistics
import pandas as pd, numpy as np
import matplotlib; import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown
pd.set_option("display.max_colwidth", 400, "display.width", 220)
OUT = "/content/Pretrained_Model_CA/Comparative_Study"; CHARTS = OUT + "/charts"; RES = OUT + "/results"
for d in (CHARTS, RES): os.makedirs(d, exist_ok=True)
def table(df):   # wrapped, left-aligned HTML table for long text
    return display(HTML(df.to_html(escape=True, index=False).replace("<td>", "<td style='text-align:left;vertical-align:top;white-space:pre-wrap'>")))

## A. Research-paper comparison
Sources: Gemma paper (arXiv 2403.08295) and DeepSeek-R1 paper (arXiv 2501.12948) + official repos/model cards. Literature statements only - no experiments here.

In [ ]:
RESEARCH = [
 [
  "Research objectives",
  "Release lightweight open models (2B, 7B) built from Gemini research/technology, with both pretrained and instruction-tuned checkpoints and an open inference codebase, and analyse safety/responsible release. [GP]",
  "Test whether reasoning can be incentivised by *pure reinforcement learning* without supervised data (R1-Zero), fix its readability/language-mixing problems with a multi-stage pipeline (R1), and distil the reasoning ability into small dense models. [DP]"
 ],
 [
  "Architecture",
  "Decoder-only Transformer, 8,192-token training context. 2B: 18 layers, d_model 2048, FFN hidden 32,768, 8 heads, 1 KV head (multi-query attention), head size 256; 7B: 28 layers, d_model 3072, 16 heads/16 KV heads (multi-head). RoPE, approximate-GeGLU, RMSNorm, shared input/output embeddings, 256k vocabulary. [GP Tables 1-2]",
  "R1-Zero / R1 are trained on DeepSeek-V3-Base, a Mixture-of-Experts model (671B total, 37B activated; 128K context per the repo README); the architecture is described in the DeepSeek-V3 report, not in the R1 paper. The distilled models are *dense Qwen2.5 / Llama* networks - the 7B uses Qwen2.5-Math-7B as base. [DP, DR]"
 ],
 [
  "Training methodology",
  "Pretraining only (then alignment): 3T (2B) and 6T (7B) tokens of mostly English web text, mathematics and code; TPUv5e (512 chips for 2B, 4096 for 7B), JAX + Pathways, staged data mixture with filtering of unsafe/personal data and evaluation-set decontamination. [GP]",
  "No new pretraining. Post-training of the V3 base: R1-Zero = RL directly on the base; R1 = 4 stages (cold-start SFT -> reasoning RL -> rejection-sampling SFT -> RL for all scenarios). Distilled models: plain SFT of open base models. [DP]"
 ],
 [
  "Instruction tuning",
  "SFT on English, text-only synthetic + human prompt-response pairs, then RLHF; role/turn control tokens (<start_of_turn>, <end_of_turn>) used at both training and inference. [GP]",
  "Cold-start: thousands of long chain-of-thought examples. Later SFT: ~600k reasoning samples (rejection sampling from the RL checkpoint) + ~200k non-reasoning samples = ~800k, 2 epochs on V3-Base. Distilled checkpoints are fine-tuned on those 800k samples. [DP §2.3-2.4]"
 ],
 [
  "Reinforcement learning",
  "RLHF: Bradley-Terry reward model from human preference pairs; policy optimised with 'a novel reinforcement learning algorithm' (not specified in the paper); LLM-judge side-by-side evaluations used for tuning and to mitigate reward hacking. [GP]",
  "GRPO (Group Relative Policy Optimization): no critic; advantage A_i = (r_i - mean(r))/std(r) over a group of G sampled outputs; clipped ratio objective with KL penalty to a reference policy. Rewards: rule-based accuracy + format (<think> tags), language-consistency reward; neural reward models only in the final all-scenario stage. *Not applied to the distilled checkpoints.* [DP §2.2, 2.3]"
 ],
 [
  "Knowledge distillation",
  "Not part of the method described in the Gemma 1 paper.",
  "Central contribution: 800k R1-generated samples used for SFT of Qwen2.5-Math-1.5B/7B, Qwen2.5-14B/32B, Llama-3.1-8B and Llama-3.3-70B-Instruct. Finding: distilling beat running large-scale RL on a 32B base (Table 6). [DP §2.4, §4.1]"
 ],
 [
  "Evaluation methodology",
  "Automated academic benchmarks (MMLU 5-shot, GSM8K maj@1, HumanEval pass@1, MBPP, BBH, ...), independent HF evaluation, human side-by-side win rates vs Mistral-7B-Instruct v0.2 (~1000 instruction + ~400 safety prompts), safety benchmarks (RealToxicity, BBQ, TruthfulQA ...), memorisation tests. [GP]",
  "Reasoning/knowledge/coding benchmarks (AIME 2024, MATH-500, GPQA-Diamond, LiveCodeBench, Codeforces, MMLU, SimpleQA, AlpacaEval 2.0, ArenaHard ...) with sampling (T=0.6, top-p 0.95), up to 64 samples/question for pass@1, cons@64, max 32,768 generated tokens; zero-shot prompts (few-shot hurts). [DP §3]"
 ],
 [
  "Contributions",
  "Open pretrained + IT checkpoints and code at 2B/7B; state-of-the-art for size at release on 11/18 text tasks; detailed safety and memorisation analysis. [GP]",
  "First open research validating RL-only incentivisation of reasoning; GRPO-based pipeline; open R1, R1-Zero and six distilled dense models; report of unsuccessful attempts (PRM, MCTS). [DP]"
 ],
 [
  "Advantages",
  "Small enough for laptops/Colab; clean control-token format; good maths/code for its size (paper); permissive-for-research open weights (custom licence).",
  "Very strong maths/code reasoning for 7B (paper: MATH-500 92.8, AIME24 55.5); emergent self-verification/reflection; MIT licence; usable like any Qwen model."
 ],
 [
  "Limitations",
  "English-centric, text-only, 8k context; may produce incorrect facts; weak absolute reasoning at 2B; residual bias/toxicity/PII risks - needs use-case safety testing. [GP, GC]",
  "Prompt-sensitive (few-shot degrades), language mixing beyond EN/ZH, weaker than V3 on function calling/JSON/multi-turn (full R1), long latency from reasoning traces, SE tasks little improved; distilled models received no RL. [DP §5]"
 ]
]
table(pd.DataFrame(RESEARCH, columns=['Dimension', 'G19 Gemma (2B-IT; paper covers 2B/7B)', 'G20 DeepSeek-R1-Distill-Qwen-7B (paper covers R1-Zero, R1 and distillation)']))

## B. Technical and implementation comparison

In [ ]:
TECH = [
 [
  "Checkpoint",
  "google/gemma-2b-it (Gemma 1.0, 2B instruction-tuned)",
  "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
 ],
 [
  "Parameter count",
  "~2.51B (524,550,144 embedding + 1,981,884,416 non-embedding) [GP Table 2]",
  "7B class (HF pages show ~7.6B-8B; the notebook prints the exact count) [DC]"
 ],
 [
  "Model architecture",
  "18-layer decoder, MQA (1 KV head), RoPE, GeGLU, RMSNorm, tied embeddings [GP]",
  "Dense Qwen2-family decoder (HF tag `qwen2`), base Qwen2.5-Math-7B; SFT-distilled from R1 [DR, DC]"
 ],
 [
  "Tokenizer",
  "SentencePiece subset of Gemini tokenizer, 256,128 entries, digit splitting, byte fallback [GP]",
  "Qwen-based tokenizer, 'slightly changed' by DeepSeek [DR]; vocab size printed at runtime; adds <think></think> handling"
 ],
 [
  "Context length",
  "Trained at 8,192 tokens [GP]",
  "Value in checkpoint config.json (printed at runtime); paper evaluates with up to 32,768 generated tokens [DP]"
 ],
 [
  "Dependencies",
  "Colab: torch, transformers, accelerate (bitsandbytes for 4/8-bit). Official repo: JAX/Flax (+ sentencepiece, kagglehub ...); current main needs Python >= 3.12 [GR]",
  "torch, transformers, accelerate (bitsandbytes for 4/8-bit); README serves with vLLM or SGLang [DR]"
 ],
 [
  "Inference framework used here",
  "Hugging Face Transformers (PyTorch). Official JAX sampler (Gemma-1 era `gemma/sampler.py`, greedy argmax) described but not run in Colab",
  "Hugging Face Transformers (PyTorch)"
 ],
 [
  "Model loading",
  "AutoModelForCausalLM.from_pretrained('google/gemma-2b-it'); gated -> accept licence + HF token [GC]",
  "AutoModelForCausalLM.from_pretrained(...); public, no token needed [DC]"
 ],
 [
  "Prompt formatting",
  "<bos><start_of_turn>user\\n{text}<end_of_turn>\\n<start_of_turn>model\\n ; no system role [GC, GP]",
  "Chat template of the checkpoint; **no system prompt**; response should begin with '<think>\\n' [DR]"
 ],
 [
  "Recommended decoding",
  "Model card examples use default greedy generate",
  "temperature 0.5-0.7 (0.6), top-p 0.95 in paper evaluation; avoid greedy (repetition) [DR, DP]"
 ],
 [
  "Hardware requirements",
  "Repo README: 8 GB+ GPU RAM for 2B, 24 GB+ for 7B [GR]; weights ~5 GB in bf16 (2.5B x 2 bytes, estimate)",
  "Weights 8.61 GB + 6.62 GB = 15.2 GB safetensors [DC] -> does not fit a 15 GB T4 in fp16 with activations; 4-bit needs roughly 5-6 GB (estimate; measured in notebook)"
 ],
 [
  "Precision / quantisation",
  "Native bf16; an fp16 revision exists; int8/int4 via bitsandbytes shown on the card [GC]",
  "Shipped in BF16 (file sizes consistent with 2 bytes/param); we use 4-bit NF4 on T4, 8-bit and fp16 where memory allows"
 ],
 [
  "Memory requirements",
  "Measured by notebook (allocated / reserved / peak)",
  "Measured by notebook (allocated / reserved / peak)"
 ],
 [
  "Output style",
  "Direct answer",
  "Long chain-of-thought inside <think>...</think> then answer"
 ],
 [
  "Licence",
  "Gemma Terms of Use (custom, gated) [GC]",
  "MIT (weights + code); base model Qwen2.5 is Apache-2.0 [DR]"
 ]
]
table(pd.DataFrame(TECH, columns=['Aspect', 'G19 Gemma', 'G20 DeepSeek']))

### Literature-reported results (NOT produced by us)
Different benchmarks, shot counts, decoding and model variants - **not comparable with each other or with our experiments**. Gemma numbers are for *pretrained* checkpoints (paper Table 6), DeepSeek numbers are the paper's reported distilled-model results.

In [ ]:
LIT = [
 [
  "MMLU (5-shot)",
  "42.3",
  "64.3",
  "not reported",
  "Gemma paper Table 6 (pretrained, not -it)"
 ],
 [
  "GSM8K (maj@1)",
  "17.7",
  "46.4",
  "not reported",
  "Gemma paper Table 6"
 ],
 [
  "MATH",
  "11.8 (4-shot)",
  "24.3 (4-shot)",
  "92.8 (MATH-500 pass@1)",
  "Different benchmark subsets and protocols"
 ],
 [
  "HumanEval pass@1",
  "22.0",
  "32.3",
  "not reported",
  "Gemma paper Table 6"
 ],
 [
  "MBPP (3-shot)",
  "29.2",
  "44.4",
  "not reported",
  "Gemma paper Table 6"
 ],
 [
  "AIME 2024 pass@1",
  "not reported",
  "not reported",
  "55.5",
  "DeepSeek paper Table 5 (T=0.6, top-p 0.95, 32k tokens)"
 ],
 [
  "GPQA-Diamond pass@1",
  "not reported",
  "not reported",
  "49.1",
  "DeepSeek paper Table 5"
 ],
 [
  "LiveCodeBench pass@1",
  "not reported",
  "not reported",
  "37.6",
  "DeepSeek paper Table 5"
 ],
 [
  "Codeforces rating",
  "not reported",
  "not reported",
  "1189",
  "DeepSeek paper Table 5"
 ],
 [
  "Human win rate vs Mistral-7B-Instruct v0.2",
  "IF 41.6% / safety 56.5% (Gemma 1.0 IT 2B)",
  "IF 51.7% / safety 58% (Gemma 1.0 IT 7B)",
  "not applicable",
  "Gemma paper Table 9"
 ]
]
table(pd.DataFrame(LIT, columns=['Benchmark', 'Gemma 2B (pretrained)', 'Gemma 7B (pretrained)', 'DeepSeek-R1-Distill-Qwen-7B', 'Source / caveat']))

## C. Experimental comparison (same held-out prompts, real prediction files)

In [ ]:
SEARCH_DIRS = ["/content", "/content/Pretrained_Model_CA/G19_Gemma/results", "/content/Pretrained_Model_CA/G20_DeepSeek/results", "/content/drive/MyDrive/Pretrained_Model_CA"]
def find(name):
    for d in SEARCH_DIRS:
        hits = glob.glob(os.path.join(d, "**", name), recursive=True)
        if hits: return hits[0]
paths = {k: find(f"{k}_results.json") for k in ("G19", "G20")}
if not all(paths.values()):
    try:
        from google.colab import files
        print("Missing:", [k for k, v in paths.items() if not v], "- upload the *_results.json file(s) now:")
        up = files.upload()
        for fn in up: os.rename(fn, f"/content/{fn}")
        paths = {k: find(f"{k}_results.json") for k in ("G19", "G20")}
    except Exception: pass
R = {k: json.load(open(p)) for k, p in paths.items() if p}
print({k: paths[k] or "NOT FOUND" for k in paths})
HAVE = len(R) == 2
if not HAVE:
    display(Markdown("### EXPERIMENTS NOT RUN / FILES MISSING\nRun the individual notebooks first. Cells below are skipped; **no values are invented**."))
for k, r in R.items():
    if r["meta"].get("quick_run"): display(Markdown(f"**WARNING: {k} results come from QUICK_RUN (reduced samples) - not valid for comparison.**"))

### C1. Same held-out test prompts?  Fairness audit

In [ ]:
if HAVE:
    ids = {k: [r["id"] for r in R[k]["test"]["records"]] for k in R}
    prompts = {k: [r["prompt"] for r in R[k]["test"]["records"]] for k in R}
    print("identical test ids:", ids["G19"] == ids["G20"], "| identical prompt text:", prompts["G19"] == prompts["G20"])
    m = {k: R[k]["meta"] for k in R}
    fair = pd.DataFrame([
        ["Checkpoint", m["G19"]["checkpoint"], m["G20"]["checkpoint"]],
        ["Parameters actually loaded (B)", round(m["G19"]["load"]["n_parameters_loaded"] / 1e9, 2), round(m["G20"]["load"]["n_parameters_loaded"] / 1e9, 2)],
        ["Precision", m["G19"]["precision"], m["G20"]["precision"]],
        ["GPU", m["G19"]["env"].get("gpu"), m["G20"]["env"].get("gpu")],
        ["Decoding", json.dumps(m["G19"]["generation_config"]), json.dumps(m["G20"]["generation_config"])],
        ["Reasoning mode", "none (direct answer)", "long chain-of-thought inside <think>, final answer evaluated"],
        ["Answers cut by token cap (dev+test)", R["G19"]["dev"]["metrics"]["n_hit_max_tokens"] + R["G19"]["test"]["metrics"]["n_hit_max_tokens"], R["G20"]["dev"]["metrics"]["n_hit_max_tokens"] + R["G20"]["test"]["metrics"]["n_hit_max_tokens"]],
        ["Runtime/GPU same session?", "see run_utc", f"{m['G19']['run_utc']} vs {m['G20']['run_utc']}"],
    ], columns=["Factor affecting fairness", "G19", "G20"])
    table(fair)
    flags = []
    if m["G19"]["precision"] != m["G20"]["precision"]: flags.append("different numeric precision / quantisation")
    if m["G19"]["generation_config"].get("do_sample") != m["G20"]["generation_config"].get("do_sample"): flags.append("greedy vs sampling decoding")
    if m["G19"]["env"].get("gpu") != m["G20"]["env"].get("gpu"): flags.append("different GPUs -> timings/memory not strictly comparable")
    flags.append("model sizes differ (~2.5B vs ~7.6B parameters) - capability gaps are confounded with size")
    display(Markdown("**Differences that limit fairness:** " + "; ".join(flags)))

### C2. Required comparison table

In [ ]:
def lm(k, split): return R[k][split]["metrics"]
def pct(x): return "n/a" if x is None else f"{100*x:.1f}%"
def f(x, d=2): return "n/a" if x is None else f"{x:.{d}f}"
if HAVE:
    rows = []
    for split, label in (("test", "HELD-OUT TEST (n=10)"), ("dev", "DEV (n=20)")):
        a, b = lm("G19", split), lm("G20", split)
        rows += [[label, "", "", ""],
            ["Checkpoint", R["G19"]["meta"]["checkpoint"], R["G20"]["meta"]["checkpoint"], ""],
            ["Loading time (s, tokenizer+model from cache)", f(R["G19"]["meta"]["load"]["total_load_s"], 1), f(R["G20"]["meta"]["load"]["total_load_s"], 1), ""],
            ["Mean inference time (s)", f(a["time_mean_s"]), f(b["time_mean_s"]), ""],
            ["Median inference time (s)", f(a["time_median_s"]), f(b["time_median_s"]), ""],
            ["Tokens per second (aggregate)", f(a["tokens_per_sec_aggregate"], 1), f(b["tokens_per_sec_aggregate"], 1), "tokens include G20's reasoning trace"],
            ["Peak GPU memory (GB, allocated)", f(a["peak_gpu_mem_gb_max"]), f(b["peak_gpu_mem_gb_max"]), "different precision!"],
            ["Accuracy", pct(a["accuracy"]) + f" ({a['n_correct']}/{a['n']})", pct(b["accuracy"]) + f" ({b['n_correct']}/{b['n']})", f"95% CI {a['accuracy_ci95'][0]:.2f}-{a['accuracy_ci95'][1]:.2f} | {b['accuracy_ci95'][0]:.2f}-{b['accuracy_ci95'][1]:.2f}"],
            ["Exact Match (short-answer items)", pct(a["exact_match"]), pct(b["exact_match"]), "strict; verbose answers score 0"],
            ["Token F1 (short-answer items)", pct(a["token_f1"]), pct(b["token_f1"]), ""],
            ["Reasoning correctness", pct(a["reasoning_correctness"]) + f" (n={a['n_reasoning']})", pct(b["reasoning_correctness"]) + f" (n={b['n_reasoning']})", ""],
            ["Instruction compliance", pct(a["instruction_compliance"]) + f" (n={a['n_compliance']})", pct(b["instruction_compliance"]) + f" (n={b['n_compliance']})", ""],
            ["Structured output validity", pct(a["structured_validity"]) + f" (n={a['n_structured']})", pct(b["structured_validity"]) + f" (n={b['n_structured']})", ""],
            ["ROUGE-L (summaries)", f(a["rougeL"], 3), f(b["rougeL"], 3), f"n={a['n_rouge']}"],
            ["Answers hitting token cap", a["n_hit_max_tokens"], b["n_hit_max_tokens"], ""]]
    MAIN = pd.DataFrame(rows, columns=["Parameter", "G19 Gemma", "G20 DeepSeek", "Note"]); table(MAIN)
    MAIN.to_csv(f"{RES}/comparison_main_table.csv", index=False)

### C3. Category-wise comparison (dev+test pooled, 3 items per category) and which model is better

In [ ]:
if HAVE:
    def percat(k):
        recs = R[k]["dev"]["records"] + R[k]["test"]["records"]; out = {}
        for r in recs: d = out.setdefault(r["category"], [0, 0]); d[0] += int(r["eval"]["correct"]); d[1] += 1
        return out
    pa, pb = percat("G19"), percat("G20")
    cats = sorted(set(pa) | set(pb)); rows = []
    for ct in cats:
        ka, na = pa.get(ct, (0, 0)); kb, nb = pb.get(ct, (0, 0))
        verdict = "tie" if ka == kb else ("G19 Gemma ahead" if ka > kb else "G20 DeepSeek ahead")
        rows.append([ct, f"{ka}/{na}", f"{kb}/{nb}", verdict, "n=3: a 1-item gap is NOT statistically meaningful"])
    CAT = pd.DataFrame(rows, columns=["Category", "G19 correct", "G20 correct", "Observed", "Caveat"]); table(CAT)
    CAT.to_csv(f"{RES}/comparison_category_table.csv", index=False)
    display(Markdown("**How to explain each difference (write after inspecting the prediction files):** open `G19_predictions_*.json` / `G20_predictions_*.json`, read the failing items and state *why* (e.g. G20 truncated while thinking, G19 format error, arithmetic slip). The notebook does not guess causes."))
    fails = {k: {} for k in R}
    for k in R:
        for r in R[k]["dev"]["records"] + R[k]["test"]["records"]:
            if not r["eval"]["correct"]: fails[k][r["id"]] = (r["category"], (r["final_output"] or r["raw_output"])[:140].replace("\n", " "), r.get("think_closed"))
    for k in fails:
        print(f"\n{k} failed items ({len(fails[k])}):")
        for i, v in fails[k].items(): print("  ", i, v[0], "| think_closed:", v[2], "|", v[1])

### C4. Charts

In [ ]:
COL = {"G19": "#3b6ea5", "G20": "#c0643a"}
def bar2(ax, vals, title, ylabel, fmt="{:.1f}"):
    xs = ["G19 Gemma", "G20 DeepSeek"]; bars = ax.bar(xs, vals, color=[COL["G19"], COL["G20"]]); ax.set_title(title, fontsize=10); ax.set_ylabel(ylabel)
    for b_, v in zip(bars, vals): ax.text(b_.get_x() + b_.get_width() / 2, v, "n/a" if v is None else fmt.format(v), ha="center", va="bottom", fontsize=9)
if HAVE:
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
    bar2(ax[0], [R[k]["meta"]["load"]["total_load_s"] for k in ("G19", "G20")], "Model loading time", "seconds")
    bar2(ax[1], [lm(k, "test")["time_mean_s"] for k in ("G19", "G20")], "Mean inference time (held-out test)", "seconds")
    bar2(ax[2], [lm(k, "test")["peak_gpu_mem_gb_max"] for k in ("G19", "G20")], "Peak GPU memory (held-out test)", "GB", "{:.2f}")
    plt.tight_layout(); plt.savefig(f"{CHARTS}/load_time_inference_time_memory.png", dpi=150); plt.show()
    mets = [("accuracy", "Accuracy"), ("exact_match", "Exact Match"), ("reasoning_correctness", "Reasoning"), ("instruction_compliance", "Compliance"), ("structured_validity", "Struct. validity")]
    x = np.arange(len(mets)); w = 0.38; fig, ax = plt.subplots(figsize=(9, 4))
    for i, k in enumerate(("G19", "G20")): ax.bar(x + (i - .5) * w, [(lm(k, "test")[m_] or 0) for m_, _ in mets], w, label=k, color=COL[k])
    ax.set_xticks(x); ax.set_xticklabels([n for _, n in mets]); ax.set_ylim(0, 1.05); ax.set_title("Evaluation scores - held-out test (n=10)"); ax.legend(); plt.tight_layout(); plt.savefig(f"{CHARTS}/evaluation_scores_test.png", dpi=150); plt.show()
    cats = sorted(pa); x = np.arange(len(cats)); fig, ax = plt.subplots(figsize=(11, 4.2))
    for i, (k, pp) in enumerate((("G19", pa), ("G20", pb))): ax.bar(x + (i - .5) * w, [pp[c_][0] / pp[c_][1] for c_ in cats], w, label=k, color=COL[k])
    ax.set_xticks(x); ax.set_xticklabels(cats, rotation=30, ha="right"); ax.set_ylim(0, 1.05); ax.set_title("Category-wise accuracy (dev+test, 3 items/category)"); ax.legend(); plt.tight_layout(); plt.savefig(f"{CHARTS}/category_accuracy.png", dpi=150); plt.show()
    vs = sorted({v for k in R for v in R[k]["robustness"]["summary"]}); x = np.arange(len(vs)); fig, ax = plt.subplots(figsize=(10, 4))
    for i, k in enumerate(("G19", "G20")): ax.bar(x + (i - .5) * w, [R[k]["robustness"]["summary"].get(v, {}).get("accuracy", 0) for v in vs], w, label=k, color=COL[k])
    ax.set_xticks(x); ax.set_xticklabels(vs, rotation=25, ha="right"); ax.set_ylim(0, 1.05); ax.set_title("Robustness accuracy by input variation"); ax.legend(); plt.tight_layout(); plt.savefig(f"{CHARTS}/robustness.png", dpi=150); plt.show()
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    for k in R:
        ok = [r for r in R[k]["resource"]["precision"] if r.get("status") == "OK"]
        ax[0].bar([f"{k}\n{r['precision']}" for r in ok], [r["peak_mem_gb"] or 0 for r in ok], color=COL[k]); ax[1].bar([f"{k}\n{r['precision']}" for r in ok], [r["metrics"]["time_mean_s"] for r in ok], color=COL[k])
    ax[0].set_title("Resource experiment: peak GPU memory (GB)"); ax[1].set_title("Resource experiment: mean time / prompt (s)"); plt.tight_layout(); plt.savefig(f"{CHARTS}/resource_precision.png", dpi=150); plt.show()

### C5. Resource experiments side by side

In [ ]:
if HAVE:
    for k in R:
        print(k, "- precision experiment (same checkpoint, different precision):")
        display(pd.DataFrame([{"precision": r["precision"], "status": r["status"][:70], **({"load_s": round(r["load"]["model_load_s"], 1), "peak_GB": round(r["peak_mem_gb"] or 0, 2), "acc": r["metrics"]["accuracy"], "mean_s": round(r["metrics"]["time_mean_s"], 1), "tok/s": round(r["metrics"]["tokens_per_sec_aggregate"], 1)} if r["status"] == "OK" else {})} for r in R[k]["resource"]["precision"]]))
        print(k, "- input length:"); display(pd.DataFrame(R[k]["resource"]["input_length"]))

### C6. Export all numbers to Excel

In [ ]:
if HAVE:
    with pd.ExcelWriter(f"{RES}/Experimental_Results.xlsx") as xw:
        MAIN.to_excel(xw, sheet_name="main_table", index=False); CAT.to_excel(xw, sheet_name="category", index=False)
        for k in R:
            pd.DataFrame([{**{kk: r[kk] for kk in ("id", "category", "seconds", "new_tokens", "tokens_per_sec", "hit_max_tokens")}, "correct": r["eval"]["correct"], "extracted": r["eval"]["extracted"], "output": (r["final_output"] or "")[:500]} for r in R[k]["dev"]["records"] + R[k]["test"]["records"]]).to_excel(xw, sheet_name=f"{k}_predictions", index=False)
            pd.DataFrame(R[k]["robustness"]["summary"]).T.to_excel(xw, sheet_name=f"{k}_robustness")
            pd.DataFrame(R[k]["failures"]).to_excel(xw, sheet_name=f"{k}_failures", index=False)
    print("wrote", f"{RES}/Experimental_Results.xlsx")

## D. Discussion template (complete after reading the tables)
1. **Per-category winner and why** - use C3 + the printed failed items. Do not call a universal winner: the evidence is n=30 items, different sizes (~2.5B vs ~7.6B), possibly different precision/decoding (C1).
2. **Cost of reasoning** - compare tokens/s vs seconds-per-correct-answer; G20 spends many tokens thinking.
3. **Format obedience** - compliance / structured validity.
4. **Threats to validity** - small n, rule-based graders, single run for sampling-based G20, token cap, quantisation.